# QianfanOCR - 元素识别（文本 / 公式 / 表格）

本 Notebook 演示如何对裁切后的文档元素进行精确识别，包括三种子任务：

- **文本解析**：提取图片中的文字内容
- **公式解析**：将公式图片转换为 LaTeX
- **表格解析**：将表格图片转换为 HTML

## 1. 环境准备

In [ ]:
import base64
import requests

## 2. 配置参数

请将 `API_URL` 和 `API_KEY` 替换为实际的服务地址和密钥。

In [ ]:
API_URL = "http://your-api-endpoint/v1/chat/completions"
API_KEY = "your-api-key"

# 模型超参
MODEL_NAME = "ocrfp8"
MAX_TOKENS = 4096

## 3. 通用调用函数

In [ ]:
def call_ocr_api(image_path: str, prompt: str) -> str:
    """调用 QianfanOCR API 并返回模型输出。"""
    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")

    # 根据文件后缀确定 MIME 类型
    suffix = image_path.rsplit(".", 1)[-1].lower()
    mime_map = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png"}
    mime_type = mime_map.get(suffix, "image/png")

    payload = {
        "model": MODEL_NAME,
        "max_tokens": MAX_TOKENS,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{image_base64}"
                        }
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ]
    }

    response = requests.post(
        API_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {API_KEY}"
        },
        json=payload
    )

    return response.json()["choices"][0]["message"]["content"]

---

## 4. 文本解析（Text Recognition）

识别图片中的文本内容，适用于对裁切后的文本区域进行精确文字提取。

In [ ]:
text_image_path = "../images/text_block.png"

text_result = call_ocr_api(
    image_path=text_image_path,
    prompt="Please extract the text from the image."
)

print("=" * 60)
print("文本解析结果：")
print("=" * 60)
print(text_result)

---

## 5. 公式解析（Formula Recognition）

将图片中的数学公式转换为 LaTeX 格式，适用于对裁切后的公式区域进行精确识别。

In [ ]:
formula_image_path = "../images/formula_block.png"

formula_result = call_ocr_api(
    image_path=formula_image_path,
    prompt="Please convert the formula in the image to LaTeX."
)

print("=" * 60)
print("公式解析结果（LaTeX）：")
print("=" * 60)
print(formula_result)

---

## 6. 表格解析（Table Parsing）

将文档图片中的表格转换为 HTML 格式，保留行列结构、合并单元格等信息。

In [ ]:
table_image_path = "../images/table_block.png"

table_result = call_ocr_api(
    image_path=table_image_path,
    prompt="Please convert the table in the image to HTML."
)

print("=" * 60)
print("表格解析结果（HTML）：")
print("=" * 60)
print(table_result)

### 表格 HTML 渲染预览

在 Jupyter Notebook 中可以直接渲染 HTML 表格。

In [ ]:
from IPython.display import HTML

HTML(table_result)